# 12 — Advanced Submission Optimization

## Araştırma protokolü

Baseline, `11_submission_optimization.ipynb` tarafından seçilen sistemdir:

- Known-TWS: `09` LightGBM uzmanı.
- Masked-TWS: `%65` `09` masked LightGBM + `%35` causal location-climatology feature'lı LightGBM.
- Temporal validation: aynı iki geç 18-tarihli expanding-window fold.
- Public leaderboard referansı: **0.824463818**; model seçiminde kullanılmaz.

Bu notebook aynı OOF satırlarında feature ablation, targeted feature aileleri, safe history, cascade, yakın model/parametre varyasyonları, seed stability, OOF ensemble ve cross-fold post-processing dener. Final seçim için local ortalama, her fold, masked performans, standart sapma, karmaşıklık ve leakage riski birlikte değerlendirilir.

In [1]:
from pathlib import Path
from time import perf_counter
import warnings, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown
from sklearn.metrics import root_mean_squared_error, mean_absolute_error
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor

warnings.filterwarnings("ignore")
SEEDS=[42,123,2026]
SEED=42
np.random.seed(SEED)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns",120)

def find_root(start=None):
    p=Path(start or Path.cwd()).resolve()
    for c in [p,*p.parents]:
        if (c/"data"/"raw"/"Train.csv").exists(): return c
    raise FileNotFoundError("Repository root bulunamadı")
ROOT=find_root(); DATA=ROOT/"data"/"raw"; OUT=ROOT/"submissions"
print(ROOT)

C:\global-water-storage-forecasting


## 1. Veri, şema ve `11` baseline denetimi

In [2]:
def load_data():
    tr=pd.read_csv(DATA/"Train.csv",parse_dates=["time"])
    te=pd.read_csv(DATA/"Test.csv",parse_dates=["time"])
    sa=pd.read_csv(DATA/"SampleSubmission.csv")
    te["_original_order"]=np.arange(len(te))
    return tr,te,sa

def validate_schema(tr,te,sa):
    assert {"sample_id","time","lat","lon","target"}.issubset(tr)
    assert {"ID","time","lat","lon","TWS_t","TWS_t_masked"}.issubset(te)
    assert list(sa.columns)==["ID","Target"]
    assert tr.sample_id.is_unique and te.ID.is_unique and sa.ID.is_unique
    assert len(te)==len(sa) and te.ID.equals(sa.ID)

train,test,sample=load_data(); validate_schema(train,test,sample)
print(train.shape,test.shape,"masked share",test.TWS_t.isna().mean())

(2154021, 13) (280961, 14) masked share 0.6652631504016572


In [3]:
SPEI=["SPEI_01_t","SPEI_03_t","SPEI_06_t","SPEI_12_t"]
def build_base_features(df):
    o=df.copy(); dt=pd.to_datetime(o.time)
    o["month"]=dt.dt.month.astype("int8"); o["quarter"]=dt.dt.quarter.astype("int8")
    o["year"]=dt.dt.year.astype("int16")
    o["elapsed_month"]=((dt.dt.year-train.time.dt.year.min())*12+dt.dt.month-1).astype("int16")
    o["month_sin_calc"]=np.sin(2*np.pi*o.month/12); o["month_cos_calc"]=np.cos(2*np.pi*o.month/12)
    o["season_sin"]=np.sin(2*np.pi*o.quarter/4); o["season_cos"]=np.cos(2*np.pi*o.quarter/4)
    s=o[SPEI]; o["spei_mean"]=s.mean(axis=1); o["spei_min"]=s.min(axis=1)
    o["spei_max"]=s.max(axis=1); o["spei_std"]=s.std(axis=1)
    o["spei_01_minus_12"]=o.SPEI_01_t-o.SPEI_12_t; o["spei_03_minus_12"]=o.SPEI_03_t-o.SPEI_12_t
    o["spei_short_long_ratio"]=o.SPEI_01_t/(o.SPEI_12_t.abs()+.1)
    o["lat_band_5"]=np.floor((o.lat+90)/5); o["lon_band_5"]=np.floor((o.lon+180)/5)
    o["spatial_bin_10"]=o.lat_band_5*100+o.lon_band_5; o["lat_lon_interaction"]=o.lat*o.lon/1000
    o["lat_sin"]=np.sin(np.radians(o.lat)); o["lat_cos"]=np.cos(np.radians(o.lat))
    o["lon_sin"]=np.sin(np.radians(o.lon)); o["lon_cos"]=np.cos(np.radians(o.lon))
    return o

def build_experimental_features(df):
    o=build_base_features(df)
    o["abs_lat"]=o.lat.abs(); o["lat_sq"]=(o.lat/90)**2; o["lon_sq"]=(o.lon/180)**2
    o["hemisphere"]=(o.lat>=0).astype("int8")
    o["month_x_hemisphere_sin"]=o.month_sin_calc*np.where(o.lat>=0,1,-1)
    o["season_x_lat"]=o.season_sin*o.lat_sin
    o["spei_median"]=o[SPEI].median(axis=1)
    o["spei_short_mean"]=o[["SPEI_01_t","SPEI_03_t"]].mean(axis=1)
    o["spei_long_mean"]=o[["SPEI_06_t","SPEI_12_t"]].mean(axis=1)
    o["spei_short_minus_long"]=o.spei_short_mean-o.spei_long_mean
    o["spei_negative_count"]=o[SPEI].lt(0).sum(axis=1).astype("int8")
    o["drought_intensity"]=(-o[SPEI].clip(upper=0)).mean(axis=1)
    o["spei_disagreement"]=o.spei_max-o.spei_min
    o["soil_x_drought"]=o.SOIL_MOISTURE_t*o.drought_intensity
    o["soil_minus_spei"]=o.SOIL_MOISTURE_t-o.spei_mean
    o["tws_x_spei06"]=o.TWS_t*o.SPEI_06_t
    o["tws_x_soil"]=o.TWS_t*o.SOIL_MOISTURE_t
    assert not np.isinf(o.select_dtypes("number")).any().any()
    return o

train_f=build_experimental_features(train); test_f=build_experimental_features(test)
GROUPS={
 "raw_climate":SPEI+["SOIL_MOISTURE_t"],
 "temporal":["month","year","quarter","elapsed_month"],
 "cyclic":["month_sin","month_cos","month_sin_calc","month_cos_calc","season_sin","season_cos"],
 "spatial":["lat","lon","lat_band_5","lon_band_5","spatial_bin_10","lat_lon_interaction","lat_sin","lat_cos","lon_sin","lon_cos"],
 "statistical":["spei_mean","spei_min","spei_max","spei_std"],
 "spei_engineered":["spei_01_minus_12","spei_03_minus_12","spei_short_long_ratio"],
 "TWS_related":["TWS_t"],
}
# 11 notebook ile yalnız set değil, kolon sırası da birebir aynıdır.
BASE_FEATURES=["lat","lon","TWS_t",*SPEI,"SOIL_MOISTURE_t","month","month_sin","month_cos","year",
 "quarter","elapsed_month","month_sin_calc","month_cos_calc","season_sin","season_cos",
 "spei_mean","spei_min","spei_max","spei_std","spei_01_minus_12","spei_03_minus_12","spei_short_long_ratio",
 "lat_band_5","lon_band_5","spatial_bin_10","lat_lon_interaction","lat_sin","lat_cos","lon_sin","lon_cos"]
BASE_MASKED=[c for c in BASE_FEATURES if c!="TWS_t"]
TARGETED={
 "targeted_temporal":["abs_lat","hemisphere","month_x_hemisphere_sin","season_x_lat"],
 "targeted_spatial":["abs_lat","lat_sq","lon_sq","hemisphere"],
 "targeted_drought":["spei_median","spei_short_mean","spei_long_mean","spei_short_minus_long",
                     "spei_negative_count","drought_intensity","spei_disagreement"],
 "targeted_interactions":["soil_x_drought","soil_minus_spei"],
}
KNOWN_INTERACTIONS=["tws_x_spei06","tws_x_soil"]
PARAMS=dict(n_estimators=160,learning_rate=.08,num_leaves=63,min_child_samples=100,
            reg_lambda=1.0,subsample=.85,colsample_bytree=.9,random_state=SEED,n_jobs=4,verbosity=-1)
print(len(BASE_FEATURES),len(BASE_MASKED),{k:len(v) for k,v in TARGETED.items()})

33 32 {'targeted_temporal': 4, 'targeted_spatial': 4, 'targeted_drought': 7, 'targeted_interactions': 2}


In [4]:
def create_temporal_folds(df,n_splits=3,block=18):
    dates=np.array(sorted(df.time.unique())); start=len(dates)-n_splits*block; out=[]
    for i in range(n_splits):
        vd=dates[start+i*block:start+(i+1)*block]; td=dates[:start+i*block]
        out.append((np.flatnonzero(df.time.isin(td)),np.flatnonzero(df.time.isin(vd))))
    return out[-2:]

date_stats=test.groupby("time").agg(n=("ID","size"),known=("TWS_t","count"))
KNOWN_POS=set(np.flatnonzero(date_stats.known.to_numpy()>date_stats.n.to_numpy()*.5))
FOLDS=[]
for fold,(ti,vi) in enumerate(create_temporal_folds(train_f),1):
    tr,va=train_f.iloc[ti].copy(),train_f.iloc[vi].copy()
    dates=np.array(sorted(va.time.unique()))
    known=va.time.isin({d for j,d in enumerate(dates) if j in KNOWN_POS}).to_numpy()
    FOLDS.append({"fold":fold,"train":tr,"valid":va,"known":known})
    print(fold,tr.time.min(),tr.time.max(),va.time.min(),va.time.max(),known.sum(),(~known).sum())

1 2002-05-01 00:00:00 2011-05-01 00:00:00 2011-08-01 00:00:00 2013-07-01 00:00:00 93511 187425


2 2002-05-01 00:00:00 2013-07-01 00:00:00 2013-11-01 00:00:00 2015-08-01 00:00:00 93638 187424


### `11` sistemi denetim özeti

| Bileşen | `11` seçimi |
|---|---|
| Known uzman | `09` LightGBM, değişmedi |
| Masked uzman 1 | `09` masked LightGBM |
| Masked uzman 2 | Aynı LightGBM + location mean/std, location-month ve global-month causal climatology |
| Blend | `%65` uzman 1 + `%35` climatology uzmanı |
| Yeni feature | Dört safe location climatology prior |
| Elenenler | Direct last/recent/month priors, persistence/all-history setleri, static interactions, XGB/CatBoost ve yakın LGBM ayarları |
| Local sonuç | Overall `0.713889 → 0.711647`; masked `0.761184 → 0.758030` |
| Public sonuç | `0.835133802 → 0.824463818` |

Public kazanımın olası açıklaması, testin çoğunluğunu oluşturan masked satırlarda düşük varyanslı location climatology sinyalinin baseline hatalarını çeşitlendirmesidir. Bu bir nedensellik kanıtı değildir.

## 2. Safe history feature cache ve `11` baseline OOF

In [5]:
HISTORY=["prior_location_mean","prior_location_std","prior_location_month","prior_global_month"]
def build_history_features_safely(history,apply):
    h=history.sort_values(["lat","lon","time"]).copy()
    loc=h.groupby(["lat","lon"]).target.agg(["mean","std"]).rename(
        columns={"mean":"prior_location_mean","std":"prior_location_std"})
    lm=h.groupby(["lat","lon","month"]).target.agg(["sum","count"]).rename(columns={"sum":"lm_sum","count":"lm_count"})
    gm=h.groupby("month").target.mean().rename("prior_global_month")
    a=apply.join(loc,on=["lat","lon"]).join(lm,on=["lat","lon","month"]).join(gm,on="month")
    a["prior_location_month"]=(a.lm_sum+4*a.prior_location_mean)/(a.lm_count.fillna(0)+4)
    return a.drop(columns=["lm_sum","lm_count"],errors="ignore")

def causal_history_for_training(df):
    o=df.sort_values(["lat","lon","time"]).copy(); shifted=o.groupby(["lat","lon"]).target.shift(1)
    o["prior_location_mean"]=shifted.groupby([o.lat,o.lon]).expanding().mean().reset_index(level=[0,1],drop=True)
    o["prior_location_std"]=shifted.groupby([o.lat,o.lon]).expanding().std().reset_index(level=[0,1],drop=True)
    o["prior_location_month"]=(shifted.groupby([o.lat,o.lon,o.month]).expanding().mean()
                                .reset_index(level=[0,1,2],drop=True))
    o["prior_global_month"]=o.groupby("month").target.transform(lambda x:x.shift(1).expanding().mean())
    return o.sort_index()

for f in FOLDS:
    f["train_hist"]=causal_history_for_training(f["train"])
    f["valid_hist"]=build_history_features_safely(f["train"],f["valid"])

def model_lgbm(params=None,seed=SEED):
    p=PARAMS.copy(); p["random_state"]=seed
    if params:p.update(params)
    return LGBMRegressor(**p)

def train_segment_model(model,train_df,valid_df,features,mask):
    t=perf_counter(); model.fit(train_df[features],train_df.target)
    return model.predict(valid_df.loc[mask,features]),perf_counter()-t,model

def oof_frame(f,pred,mask,segment,name):
    v=f["valid"].loc[mask,["sample_id","time","lat","lon","target"]].copy()
    v["prediction"]=pred; v["segment"]=segment; v["fold"]=f["fold"]; v["experiment"]=name
    v["absolute_error"]=(v.target-v.prediction).abs(); v["squared_error"]=(v.target-v.prediction)**2
    return v

# Known baseline bir kez cache'lenir.
KNOWN_PARTS=[]; KNOWN_MODELS=[]
for f in FOLDS:
    p,_,m=train_segment_model(model_lgbm(),f["train"],f["valid"],BASE_FEATURES,f["known"])
    KNOWN_PARTS.append(oof_frame(f,p,f["known"],"known","11 baseline")); KNOWN_MODELS.append(m)
KNOWN_OOF=pd.concat(KNOWN_PARTS,ignore_index=True)

def generate_masked_oof(name,features,history=False,factory=None):
    parts=[]; models=[]; sec=0
    for f in FOLDS:
        tr=f["train_hist"] if history else f["train"]; va=f["valid_hist"] if history else f["valid"]
        p,t,m=train_segment_model((factory or model_lgbm)(),tr,va,features,~f["known"])
        parts.append(oof_frame(f,p,~f["known"],"masked",name)); models.append(m); sec+=t
    return pd.concat(parts,ignore_index=True),models,sec

base_masked,base_models,_=generate_masked_oof("base masked",BASE_MASKED)
clim_masked,clim_models,_=generate_masked_oof("climatology masked",BASE_MASKED+HISTORY,True)
baseline_masked=base_masked.copy()
baseline_masked["prediction"]=.65*base_masked.prediction.to_numpy()+.35*clim_masked.prediction.to_numpy()
baseline_masked["experiment"]="11 baseline"
baseline_masked["absolute_error"]=(baseline_masked.target-baseline_masked.prediction).abs()
baseline_masked["squared_error"]=(baseline_masked.target-baseline_masked.prediction)**2
BASELINE_OOF=pd.concat([KNOWN_OOF,baseline_masked],ignore_index=True)

In [6]:
def evaluate_predictions(oof,name,complexity="medium",risk="low"):
    rows=[]
    for fold,g in oof.groupby("fold"):
        rows.append({"fold":fold,"rmse":root_mean_squared_error(g.target,g.prediction),
                     "mae":mean_absolute_error(g.target,g.prediction)})
    fr=pd.DataFrame(rows)
    known=oof[oof.segment=="known"]; masked=oof[oof.segment=="masked"]
    return {"Experiment":name,"Known RMSE":root_mean_squared_error(known.target,known.prediction),
      "Masked RMSE":root_mean_squared_error(masked.target,masked.prediction),
      "Overall RMSE":root_mean_squared_error(oof.target,oof.prediction),
      "MAE":mean_absolute_error(oof.target,oof.prediction),
      "Fold 1 RMSE":fr.loc[fr.fold==1,"rmse"].iloc[0],"Fold 2 RMSE":fr.loc[fr.fold==2,"rmse"].iloc[0],
      "Std":fr.rmse.std(),"Complexity":complexity,"Leakage risk":risk}

BASE_METRICS=evaluate_predictions(BASELINE_OOF,"11 baseline")
display(pd.DataFrame([BASE_METRICS]))
print("Expected 11 overall≈0.711647, reproduced:",BASE_METRICS["Overall RMSE"])
assert abs(BASE_METRICS["Overall RMSE"]-0.7116471792470269)<2e-6

,Experiment,Known RMSE,Masked RMSE,Overall RMSE,MAE,Fold 1 RMSE,Fold 2 RMSE,Std,Complexity,Leakage risk
0,11 baseline,0.608198,0.75803,0.711647,0.538465,0.648191,0.769866,0.086037,medium,low


Expected 11 overall≈0.711647, reproduced: 0.7116471792470269


## 3. Known uzman kontrolü ve feature ablation

In [7]:
EXPERIMENTS=[{"metrics":BASE_METRICS,"oof":BASELINE_OOF,"kind":"baseline"}]

def combine_known_masked(known,masked,name):
    o=pd.concat([known.copy(),masked.copy()],ignore_index=True); o["experiment"]=name
    return o

# Known: yalnız iki güvenli TWS interaction; mevcut güçlü sinyali bozmazsa tutulabilir.
known_int_parts=[]
for f in FOLDS:
    p,_,_=train_segment_model(model_lgbm(),f["train"],f["valid"],BASE_FEATURES+KNOWN_INTERACTIONS,f["known"])
    known_int_parts.append(oof_frame(f,p,f["known"],"known","Known TWS interactions"))
known_int=pd.concat(known_int_parts,ignore_index=True)
o=combine_known_masked(known_int,baseline_masked,"Known TWS interactions")
EXPERIMENTS.append({"metrics":evaluate_predictions(o,"Known TWS interactions","low","low"),"oof":o,"kind":"known"})

# 11 blend'in iki masked bileşeninden aynı grup çıkarılır.
ABLATION_GROUPS={k:[x for x in v if x!="TWS_t"] for k,v in GROUPS.items() if k!="TWS_related"}
for group,cols in ABLATION_GROUPS.items():
    fb=[x for x in BASE_MASKED if x not in cols]
    fc=fb+HISTORY
    b,_,_=generate_masked_oof("ablate_"+group,fb)
    c,_,_=generate_masked_oof("ablate_"+group+"_clim",fc,True)
    m=b.copy(); m["prediction"]=.65*b.prediction.to_numpy()+.35*c.prediction.to_numpy()
    o=combine_known_masked(KNOWN_OOF,m,"Ablate "+group)
    EXPERIMENTS.append({"metrics":evaluate_predictions(o,"Ablate "+group,"medium","low"),"oof":o,"kind":"ablation"})

ablation_table=pd.DataFrame([e["metrics"] for e in EXPERIMENTS if e["kind"] in ["baseline","ablation"]])
ablation_table["Delta vs baseline"]=BASE_METRICS["Overall RMSE"]-ablation_table["Overall RMSE"]
display(ablation_table.sort_values("Overall RMSE"))

,Experiment,Known RMSE,Masked RMSE,Overall RMSE,MAE,Fold 1 RMSE,Fold 2 RMSE,Std,Complexity,Leakage risk,Delta vs baseline
0,11 baseline,0.608198,0.758030,0.711647,0.538465,0.648191,0.769866,0.086037,medium,low,0.000000
4,Ablate spatial,0.608198,0.758580,0.712038,0.535977,0.672106,0.749829,0.054959,medium,low,-0.000390
6,Ablate spei_engineered,0.608198,0.760087,0.713109,0.539571,0.650024,0.771024,0.085560,medium,low,-0.001462
3,Ablate cyclic,0.608198,0.760564,0.713448,0.541000,0.651976,0.770004,0.083458,medium,low,-0.001801
5,Ablate statistical,0.608198,0.761795,0.714323,0.541028,0.653484,0.770348,0.082635,medium,low,-0.002676
1,Ablate raw_climate,0.608198,0.764946,0.716566,0.542907,0.655057,0.773172,0.083520,medium,low,-0.004919
2,Ablate temporal,0.608198,0.876423,0.797189,0.625384,0.746545,0.844782,0.069464,medium,low,-0.085542


## 4. Targeted feature aileleri ve yakın model varyasyonları

In [8]:
for family,extra in TARGETED.items():
    b,_,_=generate_masked_oof(family,BASE_MASKED+extra)
    c,_,_=generate_masked_oof(family+"_clim",BASE_MASKED+extra+HISTORY,True)
    m=b.copy(); m["prediction"]=.65*b.prediction.to_numpy()+.35*c.prediction.to_numpy()
    o=combine_known_masked(KNOWN_OOF,m,family)
    EXPERIMENTS.append({"metrics":evaluate_predictions(o,family,"medium","low"),"oof":o,"kind":"feature"})

FACTORIES={
 "Masked LGBM regularized":lambda:model_lgbm(dict(n_estimators=220,learning_rate=.06,num_leaves=31,
     min_child_samples=160,reg_alpha=.2,reg_lambda=3.0)),
 "Masked LGBM flexible":lambda:model_lgbm(dict(n_estimators=200,learning_rate=.06,num_leaves=95,
     min_child_samples=120,reg_alpha=.1,reg_lambda=2.0)),
 "Masked XGBoost":lambda:XGBRegressor(n_estimators=150,learning_rate=.07,max_depth=7,min_child_weight=30,
     subsample=.85,colsample_bytree=.9,reg_alpha=.1,reg_lambda=2,tree_method="hist",random_state=SEED,n_jobs=4),
 "Masked CatBoost":lambda:CatBoostRegressor(iterations=180,learning_rate=.08,depth=8,l2_leaf_reg=8,
     random_strength=.5,loss_function="RMSE",verbose=False,random_seed=SEED,allow_writing_files=False,thread_count=4)
}
for name,factory in FACTORIES.items():
    m,_,_=generate_masked_oof(name,BASE_MASKED,False,factory)
    o=combine_known_masked(KNOWN_OOF,m,name)
    EXPERIMENTS.append({"metrics":evaluate_predictions(o,name,"medium","low"),"oof":o,"kind":"model"})

display(pd.DataFrame([e["metrics"] for e in EXPERIMENTS if e["kind"] in ["feature","model"]]).sort_values("Overall RMSE"))

,Experiment,Known RMSE,Masked RMSE,Overall RMSE,MAE,Fold 1 RMSE,Fold 2 RMSE,Std,Complexity,Leakage risk
2,targeted_drought,0.608198,0.756693,0.710697,0.537692,0.648688,0.767689,0.084147,medium,low
1,targeted_spatial,0.608198,0.759384,0.712609,0.539437,0.649265,0.770740,0.085895,medium,low
3,targeted_interactions,0.608198,0.761227,0.713919,0.540484,0.652557,0.770385,0.083317,medium,low
4,Masked LGBM regularized,0.608198,0.763298,0.715393,0.542810,0.650296,0.775015,0.088190,medium,low
0,targeted_temporal,0.608198,0.763362,0.715439,0.542273,0.654850,0.771258,0.082313,medium,low
5,Masked LGBM flexible,0.608198,0.765692,0.717097,0.542791,0.652734,0.776115,0.087244,medium,low
6,Masked XGBoost,0.608198,0.769943,0.720127,0.546021,0.651560,0.782683,0.092718,medium,low
7,Masked CatBoost,0.608198,0.770761,0.720711,0.545982,0.654064,0.781669,0.090230,medium,low


## 5. Masked-TWS temporal cascade

In [9]:
CASCADE_FEATURES=BASE_MASKED
def temporal_tws_oof_for_training(tr):
    dates=np.array(sorted(tr.time.unique())); blocks=np.array_split(dates,4); pieces=[]
    for i in range(1,4):
        fit=tr.time.isin(np.concatenate(blocks[:i])); val=tr.time.isin(blocks[i])
        m=model_lgbm(dict(n_estimators=120)).fit(tr.loc[fit,CASCADE_FEATURES],tr.loc[fit,"TWS_t"])
        p=m.predict(tr.loc[val,CASCADE_FEATURES])
        z=tr.loc[val].copy(); z["TWS_t_estimated"]=p; pieces.append(z)
    return pd.concat(pieces).sort_index()

cascade_parts=[]; tws_scores=[]
for f in FOLDS:
    train_stage_b=temporal_tws_oof_for_training(f["train"])
    stage_a=model_lgbm(dict(n_estimators=120)).fit(f["train"][CASCADE_FEATURES],f["train"].TWS_t)
    va=f["valid"].copy(); va["TWS_t_estimated"]=stage_a.predict(va[CASCADE_FEATURES])
    tws_scores.append(root_mean_squared_error(va.TWS_t,va.TWS_t_estimated))
    stage_b=model_lgbm().fit(train_stage_b[BASE_MASKED+["TWS_t_estimated"]],train_stage_b.target)
    p=stage_b.predict(va.loc[~f["known"],BASE_MASKED+["TWS_t_estimated"]])
    cascade_parts.append(oof_frame(f,p,~f["known"],"masked","Cascade"))
cascade_masked=pd.concat(cascade_parts,ignore_index=True)
cascade_oof=combine_known_masked(KNOWN_OOF,cascade_masked,"Cascade")
EXPERIMENTS.append({"metrics":evaluate_predictions(cascade_oof,"Cascade","high","medium"),"oof":cascade_oof,"kind":"cascade"})

# Cascade + 11 baseline, basit 0.1 grid.
best=None
for w in np.arange(0,1.01,.1):
    m=baseline_masked.copy(); m["prediction"]=w*baseline_masked.prediction.to_numpy()+(1-w)*cascade_masked.prediction.to_numpy()
    o=combine_known_masked(KNOWN_OOF,m,f"Cascade blend {w:.1f}")
    met=evaluate_predictions(o,f"Cascade blend {w:.1f}","high","medium")
    if best is None or met["Overall RMSE"]<best["metrics"]["Overall RMSE"]:best={"metrics":met,"oof":o,"kind":"cascade_blend","weight":w}
EXPERIMENTS.append(best)
print("TWS estimation fold RMSE:",tws_scores)
display(pd.DataFrame([e["metrics"] for e in EXPERIMENTS if e["kind"].startswith("cascade")]))

TWS estimation fold RMSE: [0.6840253142648518, 0.7867964928140464]


,Experiment,Known RMSE,Masked RMSE,Overall RMSE,MAE,Fold 1 RMSE,Fold 2 RMSE,Std,Complexity,Leakage risk
0,Cascade,0.608198,0.763555,0.715576,0.538287,0.645495,0.779353,0.094652,high,medium
1,Cascade blend 0.6,0.608198,0.754596,0.709208,0.535154,0.642238,0.770352,0.090590,high,medium


## 6. OOF correlation, blend ve cross-fold post-processing

In [10]:
def aligned_matrix(candidates):
    ref=BASELINE_OOF[["sample_id","target","segment","fold","time","lat","lon"]].copy()
    P=[]
    for e in candidates:
        z=ref[["sample_id"]].merge(e["oof"][["sample_id","prediction"]],on="sample_id",how="left")
        assert z.prediction.notna().all(); P.append(z.prediction.to_numpy())
    return ref,np.column_stack(P)

ranked=sorted(EXPERIMENTS,key=lambda e:e["metrics"]["Overall RMSE"])
pool=[]
for e in ranked:
    if all(e["metrics"]["Experiment"]!=x["metrics"]["Experiment"] for x in pool):pool.append(e)
    if len(pool)==5:break
ref,P=aligned_matrix(pool)
corr=pd.DataFrame(P,columns=[e["metrics"]["Experiment"] for e in pool]).corr()
display(corr)

blend_trials=[]
for i in range(len(pool)):
    for j in range(i+1,len(pool)):
        for w in np.arange(0,1.01,.1):
            pred=w*P[:,i]+(1-w)*P[:,j]
            o=ref.copy(); o["prediction"]=pred; o["absolute_error"]=(o.target-pred).abs(); o["squared_error"]=(o.target-pred)**2
            met=evaluate_predictions(o,f"Blend {w:.1f} {pool[i]['metrics']['Experiment']} + {1-w:.1f} {pool[j]['metrics']['Experiment']}","medium","low")
            blend_trials.append({"metrics":met,"oof":o,"kind":"blend","components":(pool[i],pool[j],w)})
best_blend=min(blend_trials,key=lambda e:e["metrics"]["Overall RMSE"])
EXPERIMENTS.append(best_blend)

def apply_post_processing(oof,mode):
    out=oof.copy()
    # Her fold correction'ı yalnız diğer fold residual'ından öğrenir.
    for fold in [1,2]:
        train_cal=out.fold!=fold; apply=out.fold==fold
        if mode=="segment_bias":
            bias=out.loc[train_cal].groupby("segment").apply(lambda x:(x.target-x.prediction).mean())
            out.loc[apply,"prediction"]+=out.loc[apply,"segment"].map(bias).to_numpy()
        elif mode=="month_bias":
            tmp=out.loc[train_cal].assign(month=out.loc[train_cal,"time"].dt.month)
            bias=tmp.groupby(["segment","month"]).apply(lambda x:(x.target-x.prediction).mean())
            keys=list(zip(out.loc[apply,"segment"],out.loc[apply,"time"].dt.month))
            out.loc[apply,"prediction"]+=np.array([bias.get(k,0) for k in keys])
        elif mode=="linear":
            for seg in ["known","masked"]:
                m=train_cal&(out.segment==seg); a=apply&(out.segment==seg)
                slope,intercept=np.polyfit(out.loc[m,"prediction"],out.loc[m,"target"],1)
                out.loc[a,"prediction"]=slope*out.loc[a,"prediction"]+intercept
        elif mode=="clip":
            # Validation fold sınırları yalnız diğer fold target dağılımından öğrenilir.
            lo,hi=out.loc[train_cal,"target"].quantile([.005,.995])
            out.loc[apply,"prediction"]=out.loc[apply,"prediction"].clip(lo,hi)
    out["absolute_error"]=(out.target-out.prediction).abs(); out["squared_error"]=(out.target-out.prediction)**2
    return out

post_source=min(EXPERIMENTS,key=lambda e:e["metrics"]["Overall RMSE"])
for mode in ["segment_bias","month_bias","linear","clip"]:
    o=apply_post_processing(post_source["oof"],mode)
    EXPERIMENTS.append({"metrics":evaluate_predictions(o,"Post "+mode,"medium","low"),"oof":o,"kind":"post","source":post_source,"mode":mode})
display(pd.DataFrame([e["metrics"] for e in EXPERIMENTS if e["kind"] in ["blend","post"]]).sort_values("Overall RMSE"))

,Cascade blend 0.6,targeted_drought,11 baseline,Ablate spatial,Known TWS interactions
Cascade blend 0.6,1.000000,0.989205,0.994163,0.869826,0.990834
targeted_drought,0.989205,1.000000,0.991307,0.870513,0.987940
11 baseline,0.994163,0.991307,1.000000,0.872960,0.996598
Ablate spatial,0.869826,0.870513,0.872960,1.000000,0.870742
Known TWS interactions,0.990834,0.987940,0.996598,0.870742,1.000000


,Experiment,Known RMSE,Masked RMSE,Overall RMSE,MAE,Fold 1 RMSE,Fold 2 RMSE,Std,Complexity,Leakage risk
4,Post clip,0.607791,0.736226,0.696093,0.524291,0.642039,0.746221,0.073668,medium,low
0,Blend 0.5 Cascade blend 0.6 + 0.5 Ablate spatial,0.608198,0.736226,0.696212,0.524442,0.642296,0.746221,0.073486,medium,low
1,Post segment_bias,0.618106,0.785538,0.734036,0.555370,0.663070,0.798693,0.095900,medium,low
2,Post month_bias,0.649447,0.775511,0.735933,0.555460,0.665225,0.800391,0.095577,medium,low
3,Post linear,0.638161,0.793936,0.745685,0.565381,0.664708,0.818659,0.108859,medium,low


## 7. Seed stability ve robust seçim

In [11]:
pre_seed_best=min([e for e in EXPERIMENTS if e["kind"] not in ["post","blend"]],key=lambda e:e["metrics"]["Overall RMSE"])
# Seed testi: aday pipeline karmaşıksa temel masked bileşen üzerinde; 11 baseline mutlaka referans.
seed_rows=[]; seed_oofs=[]
for seed in SEEDS:
    parts=[]
    for f in FOLDS:
        p,_,_=train_segment_model(model_lgbm(seed=seed),f["train"],f["valid"],BASE_MASKED,~f["known"])
        parts.append(oof_frame(f,p,~f["known"],"masked",f"seed {seed}"))
    m=pd.concat(parts,ignore_index=True); o=combine_known_masked(KNOWN_OOF,m,f"seed {seed}")
    met=evaluate_predictions(o,f"seed {seed}","low","low"); seed_rows.append(met); seed_oofs.append(o)
seed_table=pd.DataFrame(seed_rows)
display(seed_table)
print("Seed mean/std/best/worst:",seed_table["Overall RMSE"].mean(),seed_table["Overall RMSE"].std(),
      seed_table["Overall RMSE"].min(),seed_table["Overall RMSE"].max())

summary=pd.DataFrame([e["metrics"] for e in EXPERIMENTS])
summary["Delta vs 11 baseline"]=BASE_METRICS["Overall RMSE"]-summary["Overall RMSE"]
summary["Selected?"]=False

# Robustluk: anlamlı >0.0005 overall kazanım, foldlardan hiçbirinde >0.003 bozulma yok.
eligible=summary[(summary["Delta vs 11 baseline"]>0.0005)&
                 (summary["Fold 1 RMSE"]<=BASE_METRICS["Fold 1 RMSE"]+.003)&
                 (summary["Fold 2 RMSE"]<=BASE_METRICS["Fold 2 RMSE"]+.003)]
if len(eligible):
    chosen_name=eligible.sort_values(["Overall RMSE","Std"]).iloc[0].Experiment
    CHOSEN=next(e for e in EXPERIMENTS if e["metrics"]["Experiment"]==chosen_name)
else:
    CHOSEN=EXPERIMENTS[0]; chosen_name="11 baseline"
summary.loc[summary.Experiment==chosen_name,"Selected?"]=True
display(summary.sort_values("Overall RMSE"))
print("CHOSEN:",chosen_name)

,Experiment,Known RMSE,Masked RMSE,Overall RMSE,MAE,Fold 1 RMSE,Fold 2 RMSE,Std,Complexity,Leakage risk
0,seed 42,0.608198,0.761184,0.713889,0.540541,0.646900,0.775083,0.090639,low,low
1,seed 123,0.608198,0.761986,0.714460,0.541760,0.649987,0.773552,0.087374,low,low
2,seed 2026,0.608198,0.759753,0.712871,0.539557,0.644693,0.775048,0.092175,low,low


Seed mean/std/best/worst: 0.7137398734863862 0.0008046141064288709 0.7128712575103885 0.7144597223207546


,Experiment,Known RMSE,Masked RMSE,Overall RMSE,MAE,Fold 1 RMSE,Fold 2 RMSE,Std,Complexity,Leakage risk,Delta vs 11 baseline,Selected?
22,Post clip,0.607791,0.736226,0.696093,0.524291,0.642039,0.746221,0.073668,medium,low,0.015554,True
18,Blend 0.5 Cascade blend 0.6 + 0.5 Ablate spatial,0.608198,0.736226,0.696212,0.524442,0.642296,0.746221,0.073486,medium,low,0.015435,False
17,Cascade blend 0.6,0.608198,0.754596,0.709208,0.535154,0.642238,0.770352,0.090590,high,medium,0.002439,False
10,targeted_drought,0.608198,0.756693,0.710697,0.537692,0.648688,0.767689,0.084147,medium,low,0.000950,False
0,11 baseline,0.608198,0.758030,0.711647,0.538465,0.648191,0.769866,0.086037,medium,low,0.000000,False
5,Ablate spatial,0.608198,0.758580,0.712038,0.535977,0.672106,0.749829,0.054959,medium,low,-0.000390,False
1,Known TWS interactions,0.609924,0.758030,0.712139,0.539418,0.647223,0.771587,0.087939,low,low,-0.000492,False
9,targeted_spatial,0.608198,0.759384,0.712609,0.539437,0.649265,0.770740,0.085895,medium,low,-0.000962,False
7,Ablate spei_engineered,0.608198,0.760087,0.713109,0.539571,0.650024,0.771024,0.085560,medium,low,-0.001462,False
4,Ablate cyclic,0.608198,0.760564,0.713448,0.541000,0.651976,0.770004,0.083458,medium,low,-0.001801,False


CHOSEN: Post clip


## 8. Final training ve submission

In [12]:
full_train_hist=causal_history_for_training(train_f)
full_test_hist=build_history_features_safely(train_f,test_f)
known_test=test_f.TWS_t.notna().to_numpy()

def fit_known(features=BASE_FEATURES):
    m=model_lgbm().fit(train_f[features],train_f.target)
    return m.predict(test_f.loc[known_test,features])
def fit_masked_base(features=BASE_MASKED,factory=None):
    m=(factory or model_lgbm)().fit(train_f[features],train_f.target)
    return m.predict(test_f.loc[~known_test,features])
def fit_masked_clim(features=BASE_MASKED+HISTORY):
    m=model_lgbm().fit(full_train_hist[features],train_f.target)
    return m.predict(full_test_hist.loc[~known_test,features])

# Her zaman 11 baseline test tahmini yeniden üretilir.
pred11=np.empty(len(test_f)); pred11[known_test]=fit_known()
pred11[~known_test]=.65*fit_masked_base()+.35*fit_masked_clim()

# Final pipeline yeniden üretimi: karmaşık adayların tanımlı türleri desteklenir; aksi halde 11 korunur.
def train_final_models(chosen):
    name=chosen["metrics"]["Experiment"]
    if name=="11 baseline": return pred11.copy()
    # OOF blend ise bileşenler recursive; güvenli ve destekli bileşenler için tahmin.
    if chosen["kind"]=="blend":
        a,b,w=chosen["components"]
        return w*train_final_models(a)+(1-w)*train_final_models(b)
    if chosen["kind"]=="post":
        raw=train_final_models(chosen["source"])
        if chosen.get("mode")=="clip":
            lo,hi=train.target.quantile([.005,.995]); return np.clip(raw,lo,hi)
        return raw  # Diğer cross-fold düzeltmeler testte güvenli sabit taşımıyorsa uygulanmaz.
    if chosen["kind"] in ["cascade","cascade_blend"]:
        # Stage-B train feature'ı temporal OOF TWS tahminidir; in-sample TWS_estimated kullanılmaz.
        stage_train=temporal_tws_oof_for_training(train_f)
        stage_a=model_lgbm(dict(n_estimators=120)).fit(train_f[CASCADE_FEATURES],train_f.TWS_t)
        test_stage=test_f.copy(); test_stage["TWS_t_estimated"]=stage_a.predict(test_f[CASCADE_FEATURES])
        stage_b=model_lgbm().fit(stage_train[BASE_MASKED+["TWS_t_estimated"]],stage_train.target)
        cascade_pred=pred11.copy()
        cascade_pred[~known_test]=stage_b.predict(test_stage.loc[~known_test,BASE_MASKED+["TWS_t_estimated"]])
        if chosen["kind"]=="cascade_blend":
            w=chosen["weight"]; return w*pred11+(1-w)*cascade_pred
        return cascade_pred
    if chosen["kind"]=="known":
        p=pred11.copy(); p[known_test]=fit_known(BASE_FEATURES+KNOWN_INTERACTIONS); return p
    if chosen["kind"]=="feature":
        extra=TARGETED[name]; p=np.empty(len(test_f)); p[known_test]=pred11[known_test]
        b=fit_masked_base(BASE_MASKED+extra)
        cmodel=model_lgbm().fit(full_train_hist[BASE_MASKED+extra+HISTORY],train_f.target)
        c=cmodel.predict(full_test_hist.loc[~known_test,BASE_MASKED+extra+HISTORY])
        p[~known_test]=.65*b+.35*c; return p
    if chosen["kind"]=="ablation":
        group=name.replace("Ablate ",""); cols=ABLATION_GROUPS[group]
        fb=[x for x in BASE_MASKED if x not in cols]; p=np.empty(len(test_f)); p[known_test]=pred11[known_test]
        p[~known_test]=.65*fit_masked_base(fb)+.35*fit_masked_clim(fb+HISTORY); return p
    # Model/cascade karmaşıklığı küçük local kazanımı haklı çıkarmıyorsa baseline fallback.
    return pred11.copy()

pred12=train_final_models(CHOSEN)

In [13]:
def validate_submission(sub,test_original,sample):
    checks={
      "row_count":len(sub)==len(sample)==len(test_original),
      "same_id_set":set(sub.ID)==set(sample.ID),
      "same_id_order":sub.ID.equals(sample.ID),
      "duplicate_id":not sub.ID.duplicated().any(),
      "missing_extra_id":set(sub.ID)==set(test_original.ID),
      "columns":list(sub.columns)==list(sample.columns)==["ID","Target"],
      "numeric":pd.api.types.is_numeric_dtype(sub.Target),
      "finite":np.isfinite(sub.Target).all(),
      "known_count":int(test_original.TWS_t.notna().sum())==94048,
      "masked_count":int(test_original.TWS_t.isna().sum())==186913,
      "reasonable_range":sub.Target.between(train.target.min()-1,train.target.max()+1).all()
    }
    assert all(checks.values()),checks
    return checks

def build_submission(pred,path):
    by_id=pd.Series(np.asarray(pred,float),index=test_f.ID.astype(str))
    sub=sample.copy(); sub["Target"]=sample.ID.astype(str).map(by_id).to_numpy(float)
    checks=validate_submission(sub,test_f,sample)
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True); sub.to_csv(path,index=False)
    reread=pd.read_csv(path); validate_submission(reread,test_f,sample)
    print(path,checks); display(sub.Target.describe()); return sub

optimized_path=OUT/"submission_12_optimized.csv"
sub12=build_submission(pred12,optimized_path)

# Conservative blend oranı yalnız OOF: 11 baseline ile chosen arasında 0.1 grid.
ref2,P2=aligned_matrix([EXPERIMENTS[0],CHOSEN])
blend_grid=[]
for w in np.arange(0,1.01,.1):
    pp=w*P2[:,0]+(1-w)*P2[:,1]
    blend_grid.append((w,root_mean_squared_error(ref2.target,pp)))
best_cons=min(blend_grid,key=lambda x:x[1])
conservative_path=None
if best_cons[1]<BASE_METRICS["Overall RMSE"]-1e-6 and 0<best_cons[0]<1:
    conservative_path=OUT/"submission_12_conservative_blend.csv"
    conservative_pred=best_cons[0]*pred11+(1-best_cons[0])*pred12
    conservative_sub=build_submission(conservative_pred,conservative_path)
else:
    print("Conservative blend OOF iyileştirmedi; ikinci submission üretilmedi.",best_cons)

C:\global-water-storage-forecasting\submissions\submission_12_optimized.csv {'row_count': True, 'same_id_set': True, 'same_id_order': True, 'duplicate_id': True, 'missing_extra_id': True, 'columns': True, 'numeric': True, 'finite': np.True_, 'known_count': True, 'masked_count': True, 'reasonable_range': np.True_}


count    280961.000000
mean         -0.141210
std           0.483718
min          -2.179287
25%          -0.444157
50%          -0.179200
75%           0.132456
max           2.054586
Name: Target, dtype: float64

Conservative blend OOF iyileştirmedi; ikinci submission üretilmedi. (np.float64(0.0), 0.6960934907515215)


## 9. Final rapor

In [14]:
def summarize_experiments():
    best=CHOSEN["metrics"]; failed=summary[summary["Delta vs 11 baseline"]<=0].Experiment.tolist()
    used_post=CHOSEN["kind"]=="post"
    report=f'''
### Kapsamlı sonuç

- **11 baseline:** known `{BASE_METRICS["Known RMSE"]:.6f}`, masked `{BASE_METRICS["Masked RMSE"]:.6f}`, overall `{BASE_METRICS["Overall RMSE"]:.6f}`.
- **En iyi robust yeni sonuç:** `{chosen_name}`; known `{best["Known RMSE"]:.6f}`, masked `{best["Masked RMSE"]:.6f}`, overall `{best["Overall RMSE"]:.6f}`.
- **Değişim:** known `{BASE_METRICS["Known RMSE"]-best["Known RMSE"]:+.6f}`, masked `{BASE_METRICS["Masked RMSE"]-best["Masked RMSE"]:+.6f}`, overall `{BASE_METRICS["Overall RMSE"]-best["Overall RMSE"]:+.6f}`.
- **Foldlar/std:** `{best["Fold 1 RMSE"]:.6f}`, `{best["Fold 2 RMSE"]:.6f}`, std `{best["Std"]:.6f}`.
- **Başarılı fikirler:** {", ".join(summary.loc[summary["Delta vs 11 baseline"]>0,"Experiment"].tolist()) or "Yok"}.
- **Başarısız fikirler:** {", ".join(failed)}.
- **Ablation/eliminasyon:** Ablation tablosunda pozitif delta veren çıkarmalar sadeleştirme adayıdır; negatif delta veren gruplar korunur.
- **Cascade TWS estimation RMSE:** {", ".join(f"{x:.4f}" for x in tws_scores)}; yalnız target OOF iyileştirmesi varsa seçilebilir.
- **Seed stability:** mean `{seed_table["Overall RMSE"].mean():.6f}`, std `{seed_table["Overall RMSE"].std():.6f}`, best `{seed_table["Overall RMSE"].min():.6f}`, worst `{seed_table["Overall RMSE"].max():.6f}`.
- **Post-processing:** {"Kullanıldı: "+CHOSEN.get("mode","") if used_post else "Final pipeline'da kullanılmadı."}
- **Seçilen feature listesi:** Known `{BASE_FEATURES}`; masked baseline `{BASE_MASKED}`; history `{HISTORY}`.
- **Submission:** `{optimized_path}`. Conservative: `{conservative_path if conservative_path else "OOF desteği yoksa üretilmedi"}`.
- **Genelleme beklentisi:** Seçim test/leaderboard'a değil aynı temporal OOF satırlarına, fold tutarlılığına ve düşük leakage riskine dayanır. Known/masked ayrımı ve `11` climatology çeşitliliği korunur.
- **Kalan riskler:** Yalnız iki geç fold, test horizon düzensizliği, history feature train/deployment semantik farkı ve çoklu deney kaynaklı selection overfit. Public `0.824463818` skorunun aşılması garanti değildir.
'''
    display(Markdown(report)); return report
FINAL_REPORT=summarize_experiments()


### Kapsamlı sonuç

- **11 baseline:** known `0.608198`, masked `0.758030`, overall `0.711647`.
- **En iyi robust yeni sonuç:** `Post clip`; known `0.607791`, masked `0.736226`, overall `0.696093`.
- **Değişim:** known `+0.000407`, masked `+0.021804`, overall `+0.015554`.
- **Foldlar/std:** `0.642039`, `0.746221`, std `0.073668`.
- **Başarılı fikirler:** targeted_drought, Cascade blend 0.6, Blend 0.5 Cascade blend 0.6 + 0.5 Ablate spatial, Post clip.
- **Başarısız fikirler:** 11 baseline, Known TWS interactions, Ablate raw_climate, Ablate temporal, Ablate cyclic, Ablate spatial, Ablate statistical, Ablate spei_engineered, targeted_temporal, targeted_spatial, targeted_interactions, Masked LGBM regularized, Masked LGBM flexible, Masked XGBoost, Masked CatBoost, Cascade, Post segment_bias, Post month_bias, Post linear.
- **Ablation/eliminasyon:** Ablation tablosunda pozitif delta veren çıkarmalar sadeleştirme adayıdır; negatif delta veren gruplar korunur.
- **Cascade TWS estimation RMSE:** 0.6840, 0.7868; yalnız target OOF iyileştirmesi varsa seçilebilir.
- **Seed stability:** mean `0.713740`, std `0.000805`, best `0.712871`, worst `0.714460`.
- **Post-processing:** Kullanıldı: clip
- **Seçilen feature listesi:** Known `['lat', 'lon', 'TWS_t', 'SPEI_01_t', 'SPEI_03_t', 'SPEI_06_t', 'SPEI_12_t', 'SOIL_MOISTURE_t', 'month', 'month_sin', 'month_cos', 'year', 'quarter', 'elapsed_month', 'month_sin_calc', 'month_cos_calc', 'season_sin', 'season_cos', 'spei_mean', 'spei_min', 'spei_max', 'spei_std', 'spei_01_minus_12', 'spei_03_minus_12', 'spei_short_long_ratio', 'lat_band_5', 'lon_band_5', 'spatial_bin_10', 'lat_lon_interaction', 'lat_sin', 'lat_cos', 'lon_sin', 'lon_cos']`; masked baseline `['lat', 'lon', 'SPEI_01_t', 'SPEI_03_t', 'SPEI_06_t', 'SPEI_12_t', 'SOIL_MOISTURE_t', 'month', 'month_sin', 'month_cos', 'year', 'quarter', 'elapsed_month', 'month_sin_calc', 'month_cos_calc', 'season_sin', 'season_cos', 'spei_mean', 'spei_min', 'spei_max', 'spei_std', 'spei_01_minus_12', 'spei_03_minus_12', 'spei_short_long_ratio', 'lat_band_5', 'lon_band_5', 'spatial_bin_10', 'lat_lon_interaction', 'lat_sin', 'lat_cos', 'lon_sin', 'lon_cos']`; history `['prior_location_mean', 'prior_location_std', 'prior_location_month', 'prior_global_month']`.
- **Submission:** `C:\global-water-storage-forecasting\submissions\submission_12_optimized.csv`. Conservative: `OOF desteği yoksa üretilmedi`.
- **Genelleme beklentisi:** Seçim test/leaderboard'a değil aynı temporal OOF satırlarına, fold tutarlılığına ve düşük leakage riskine dayanır. Known/masked ayrımı ve `11` climatology çeşitliliği korunur.
- **Kalan riskler:** Yalnız iki geç fold, test horizon düzensizliği, history feature train/deployment semantik farkı ve çoklu deney kaynaklı selection overfit. Public `0.824463818` skorunun aşılması garanti değildir.


## Değişiklik özeti

- `11` baseline birebir yeniden üretildi ve assertion ile doğrulandı.
- Aynı OOF satırlarında known/masked ayrı deneyler, ablation, targeted feature, safe history, cascade, model/parametre, seed, ensemble ve cross-fold post-processing çalıştırıldı.
- Robust seçim kuralı uygulandı.
- Submission kontrolleri geçmeden dosya yazılmadı.